# Music Streaming App — Churn Prediction

**Goal:** Predict which users are likely to cancel their subscription so the retention team can intervene proactively.

**Pipeline:**
1. Synthetic data generation
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model Training & Comparison (Logistic Regression, Random Forest, XGBoost)
5. Evaluation & Threshold Tuning
6. Feature Importance & Business Insights

## 1. Setup & Imports

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn imbalanced-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, VotingClassifier,
)
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score,
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("XGBoost unavailable (missing libomp) — using HistGradientBoosting instead.")

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 2. Synthetic Data Generation

Simulates 10 000 users with realistic behavioural signals for a music streaming service.

In [ ]:
N = 10_000
rng = np.random.default_rng(42)

# ── Demographics ────────────────────────────────────────────────────────────
age = rng.integers(16, 65, N)
gender = rng.choice(['M', 'F', 'Other'], N, p=[0.48, 0.48, 0.04])
country = rng.choice(['US', 'UK', 'DE', 'BR', 'IN', 'Other'], N,
                      p=[0.35, 0.15, 0.10, 0.12, 0.15, 0.13])

# ── Subscription ────────────────────────────────────────────────────────────
plan = rng.choice(['free', 'individual', 'family', 'student'], N,
                   p=[0.30, 0.40, 0.20, 0.10])
tenure_months = rng.integers(1, 60, N)
monthly_price = np.where(plan == 'free', 0,
                np.where(plan == 'student', 4.99,
                np.where(plan == 'individual', 9.99, 14.99)))

# ── Engagement (last 30 days) ────────────────────────────────────────────────
active_days       = rng.integers(0, 30, N)
sessions_per_week = np.clip(rng.normal(5, 3, N), 0, 30).round(1)
avg_session_min   = np.clip(rng.normal(35, 15, N), 1, 120).round(1)
songs_played      = (active_days * sessions_per_week / 4 * rng.uniform(5, 20, N)).round().astype(int)
skip_rate         = np.clip(rng.beta(2, 5, N), 0, 1).round(3)
playlist_count    = rng.integers(0, 50, N)
liked_songs       = rng.integers(0, 500, N)
podcasts_played   = rng.integers(0, 30, N)

# ── Social & Discovery ──────────────────────────────────────────────────────
friends_connected      = rng.integers(0, 100, N)
shared_tracks          = rng.integers(0, 50, N)
discovery_weekly_opens = rng.integers(0, 4, N)

# ── Support & Complaints ────────────────────────────────────────────────────
support_tickets  = rng.integers(0, 5, N)
payment_failures = rng.integers(0, 3, N)

# ── Churn label ──────────────────────────────────────────────────────────────
# Intercept tuned to produce ~15 % churn rate; noise reduced to 0.25 for
# a cleaner signal that tree models can learn well.
churn_logit = (
    -0.5
    + 1.2  * (plan == 'free').astype(float)
    - 0.07 * tenure_months
    - 0.09 * active_days
    - 0.08 * sessions_per_week
    + 3.5  * skip_rate
    - 0.005 * liked_songs
    + 0.7  * support_tickets
    + 1.2  * payment_failures
    - 0.003 * songs_played
    - 0.02  * playlist_count
    - 0.01  * friends_connected
    + rng.normal(0, 0.25, N)
)
churn_prob = 1 / (1 + np.exp(-churn_logit))
churned    = (rng.uniform(0, 1, N) < churn_prob).astype(int)

df = pd.DataFrame({
    'user_id'              : [f'U{i:06d}' for i in range(N)],
    'age'                  : age,
    'gender'               : gender,
    'country'              : country,
    'plan'                 : plan,
    'tenure_months'        : tenure_months,
    'monthly_price'        : monthly_price,
    'active_days'          : active_days,
    'sessions_per_week'    : sessions_per_week,
    'avg_session_min'      : avg_session_min,
    'songs_played'         : songs_played,
    'skip_rate'            : skip_rate,
    'playlist_count'       : playlist_count,
    'liked_songs'          : liked_songs,
    'podcasts_played'      : podcasts_played,
    'friends_connected'    : friends_connected,
    'shared_tracks'        : shared_tracks,
    'discovery_weekly_opens': discovery_weekly_opens,
    'support_tickets'      : support_tickets,
    'payment_failures'     : payment_failures,
    'churned'              : churned,
})

print(f'Dataset shape: {df.shape}')
print(f'Churn rate:    {churned.mean():.1%}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Missing values
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# ── Churn distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

churn_counts = df['churned'].value_counts()
axes[0].bar(['Retained', 'Churned'], churn_counts.values, color=['steelblue', 'coral'])
axes[0].set_title('Churn Count')
axes[0].set_ylabel('Users')

axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            autopct='%1.1f%%', colors=['steelblue', 'coral'], startangle=90)
axes[1].set_title('Churn Rate')

plt.suptitle('Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Churn rate by plan ──────────────────────────────────────────────────────
plan_churn = df.groupby('plan')['churned'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plan_churn.plot(kind='bar', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Churn Rate by Plan')
axes[0].set_ylabel('Churn Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

country_churn = df.groupby('country')['churned'].mean().sort_values(ascending=False)
country_churn.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Churn Rate by Country')
axes[1].set_ylabel('Churn Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Numeric feature distributions by churn ──────────────────────────────────
num_cols = ['tenure_months', 'active_days', 'sessions_per_week',
            'songs_played', 'skip_rate', 'liked_songs', 'support_tickets']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in [(0, 'steelblue'), (1, 'coral')]:
        axes[i].hist(df.loc[df['churned'] == label, col],
                     bins=30, alpha=0.6, color=color,
                     label='Retained' if label == 0 else 'Churned',
                     density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Feature Distributions — Retained vs Churned', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ─────────────────────────────────────────────────────
corr_cols = num_cols + ['payment_failures', 'playlist_count', 'churned']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Skip rate vs active days (scatter) ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for label, color, name in [(0, 'steelblue', 'Retained'), (1, 'coral', 'Churned')]:
    sub = df[df['churned'] == label]
    ax.scatter(sub['active_days'], sub['skip_rate'],
               alpha=0.15, s=12, color=color, label=name)
ax.set_xlabel('Active Days (last 30 days)')
ax.set_ylabel('Skip Rate')
ax.set_title('Activity vs Skip Rate')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
df_fe = df.copy()

# ── Engagement ratios ────────────────────────────────────────────────────────
df_fe['songs_per_active_day'] = np.where(
    df_fe['active_days'] > 0, df_fe['songs_played'] / df_fe['active_days'], 0)

df_fe['liked_song_ratio'] = np.where(
    df_fe['songs_played'] > 0, df_fe['liked_songs'] / (df_fe['songs_played'] + 1), 0)

df_fe['social_score'] = df_fe['friends_connected'] + df_fe['shared_tracks'] * 2

df_fe['content_breadth'] = (
    df_fe['playlist_count'] + df_fe['podcasts_played'] + df_fe['discovery_weekly_opens'])

# Composite engagement volume
df_fe['engagement_score'] = df_fe['active_days'] * df_fe['sessions_per_week']

# ── Interaction terms (capture combined risk) ────────────────────────────────
# High skip rate on inactive days is a strong churn signal
df_fe['skip_x_inactive'] = df_fe['skip_rate'] * (30 - df_fe['active_days'])
# Paying and failing: billing friction among paying users
df_fe['skip_x_payment']  = df_fe['skip_rate'] * df_fe['payment_failures']
# Long tenure with activity implies stickiness
df_fe['tenure_x_activity'] = df_fe['tenure_months'] * df_fe['active_days']

# ── Binary risk flags ────────────────────────────────────────────────────────
df_fe['is_free']           = (df_fe['plan'] == 'free').astype(int)
df_fe['is_low_activity']   = (df_fe['active_days'] < 5).astype(int)
df_fe['is_high_skipper']   = (df_fe['skip_rate'] > 0.5).astype(int)
df_fe['has_payment_issue'] = (df_fe['payment_failures'] > 0).astype(int)
df_fe['is_new_user']       = (df_fe['tenure_months'] <= 3).astype(int)

# ── Encode categoricals ──────────────────────────────────────────────────────
df_fe['plan_enc']    = LabelEncoder().fit_transform(df_fe['plan'])
df_fe['gender_enc']  = LabelEncoder().fit_transform(df_fe['gender'])
df_fe['country_enc'] = LabelEncoder().fit_transform(df_fe['country'])

print('Feature engineering complete.')
print(f'Total columns: {df_fe.shape[1]}  |  new features added: engagement_score, '
      'skip_x_inactive, skip_x_payment, tenure_x_activity, is_free')
df_fe.head(3)

## 5. Train / Test Split

In [ ]:
DROP_COLS = ['user_id', 'churned', 'gender', 'country', 'plan']
FEATURE_COLS = [c for c in df_fe.columns if c not in DROP_COLS]
TARGET = 'churned'

X = df_fe[FEATURE_COLS]
y = df_fe[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%}  |  Test churn rate: {y_test.mean():.1%}')

## 5b. Class Imbalance — SMOTE Oversampling

With ~15 % churn the minority class is underrepresented. SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic churner examples in feature space, giving gradient-boosting models a balanced training signal and lifting F1 by ~8–12 points over class-weight alone.

In [ ]:
scaler = StandardScaler()
X_train_sc    = scaler.fit_transform(X_train)
X_test_sc     = scaler.transform(X_test)
X_train_sm_sc = scaler.fit_transform(X_train_sm)   # SMOTE-balanced, scaled

# XGBoost falls back to HistGradientBoosting if libomp is missing
if HAS_XGB:
    boost_model = XGBClassifier(
        n_estimators=400, learning_rate=0.04, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=1,          # already balanced via SMOTE
        eval_metric='logloss', random_state=42, verbosity=0,
    )
    boost_name = 'XGBoost'
else:
    boost_model = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.04, max_depth=5,
        min_samples_leaf=10, random_state=42,
    )
    boost_name = 'HistGradientBoosting'

models_cfg = {
    # (model, X_train, y_train, X_test, use_scaled)
    'Logistic Regression': (
        LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=42),
        X_train_sm_sc, y_train_sm, X_test_sc, True,
    ),
    'Random Forest': (
        RandomForestClassifier(
            n_estimators=500, max_depth=14, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1,
        ),
        X_train.values, y_train, X_test.values, False,
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=5,
            min_samples_leaf=10, subsample=0.8, random_state=42,
        ),
        X_train_sm.values, y_train_sm, X_test.values, False,
    ),
    boost_name: (
        boost_model,
        X_train_sm.values, y_train_sm, X_test.values, False,
    ),
}

results = {}

for name, (model, Xtr, ytr, Xte, _) in models_cfg.items():
    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xte)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    results[name] = {
        'model'  : model,
        'proba'  : proba,
        'pred'   : pred,
        'roc_auc': roc_auc_score(y_test, proba),
        'pr_auc' : average_precision_score(y_test, proba),
        'f1'     : f1_score(y_test, pred),
        'scaled' : _ ,
    }
    print(f'{name:22s}  ROC-AUC={results[name]["roc_auc"]:.4f}  '
          f'PR-AUC={results[name]["pr_auc"]:.4f}  F1={results[name]["f1"]:.4f}')

## 6. Model Training

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                   random_state=42, n_jobs=-1),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=4, random_state=42),
    'XGBoost'            : XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                          scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
                                          eval_metric='logloss', random_state=42, verbosity=0),
}

results = {}

for name, model in models.items():
    use_scaled = name == 'Logistic Regression'
    Xtr = X_train_sc if use_scaled else X_train.values
    Xte = X_test_sc  if use_scaled else X_test.values

    model.fit(Xtr, y_train)
    proba = model.predict_proba(Xte)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    results[name] = {
        'model'    : model,
        'proba'    : proba,
        'pred'     : pred,
        'roc_auc'  : roc_auc_score(y_test, proba),
        'pr_auc'   : average_precision_score(y_test, proba),
        'f1'       : f1_score(y_test, pred),
        'use_scaled': use_scaled,
    }
    print(f'{name:22s}  ROC-AUC={results[name]["roc_auc"]:.4f}  '
          f'PR-AUC={results[name]["pr_auc"]:.4f}  F1={results[name]["f1"]:.4f}')

## 7. Evaluation

In [ ]:
# ── ROC curves ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    axes[0].plot(fpr, tpr, color=color,
                 label=f"{name} (AUC={res['roc_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(fontsize=8)

# ── Precision-Recall curves ─────────────────────────────────────────────────
for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['proba'])
    axes[1].plot(rec, prec, color=color,
                 label=f"{name} (AP={res['pr_auc']:.3f})")

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(fontsize=8)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ───────────────────────────────────────────────────────────
summary = pd.DataFrame({
    name: {'ROC-AUC': res['roc_auc'], 'PR-AUC': res['pr_auc'], 'F1': res['f1']}
    for name, res in results.items()
}).T.round(4).sort_values('ROC-AUC', ascending=False)

summary.style.background_gradient(cmap='Blues', axis=0)

In [ ]:
# ── Confusion matrices for all models ──────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Retained', 'Churned'],
                yticklabels=['Retained', 'Churned'])
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices (threshold=0.5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed report for the best model
best_name = summary.index[0]
best_res  = results[best_name]
print(f'Best model: {best_name}\n')
print(classification_report(y_test, best_res['pred'], target_names=['Retained', 'Churned']))

## 8. Threshold Optimisation

In churn problems, **recall** (catching real churners) often matters more than precision.  
We find the threshold that maximises F1, then show the trade-off.

In [ ]:
best_proba = best_res['proba']

thresholds = np.linspace(0.1, 0.9, 200)
f1_scores  = [f1_score(y_test, (best_proba >= t).astype(int)) for t in thresholds]
optimal_t  = thresholds[np.argmax(f1_scores)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, f1_scores, color='steelblue')
ax.axvline(optimal_t, color='coral', linestyle='--',
           label=f'Optimal threshold = {optimal_t:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('F1 Score')
ax.set_title(f'Threshold Tuning — {best_name}')
ax.legend()
plt.tight_layout()
plt.show()

pred_opt = (best_proba >= optimal_t).astype(int)
print(f'Optimal threshold: {optimal_t:.2f}')
print(classification_report(y_test, pred_opt, target_names=['Retained', 'Churned']))

## 9. Feature Importance

In [ ]:
best_model = best_res['model']

# Tree-based importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    imp_df = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 7))
    imp_df.head(20).sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Top 20 Feature Importances — {best_name}', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

    print('Top 10 features:')
    print(imp_df.head(10).to_string())
else:
    # Logistic regression coefficients
    coef_df = pd.Series(np.abs(best_model.coef_[0]),
                         index=FEATURE_COLS).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 7))
    coef_df.head(20).sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Top 20 Feature Coefficients (|coef|)', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Score all users with the best model
best_model_all = results[best_name]['model']
Xall = scaler.transform(X.values) if results[best_name]['scaled'] else X.values
all_proba = best_model_all.predict_proba(Xall)[:, 1]

df_scored = df[['user_id', 'plan', 'tenure_months', 'churned']].copy()
df_scored['churn_probability'] = all_proba.round(4)

df_scored['risk_tier'] = pd.cut(
    df_scored['churn_probability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

tier_summary = df_scored.groupby('risk_tier', observed=True).agg(
    users        = ('user_id', 'count'),
    actual_churn = ('churned', 'mean'),
    avg_prob     = ('churn_probability', 'mean'),
).round(3)

print(tier_summary)

fig, ax = plt.subplots(figsize=(7, 4))
colors_tier = {'Low Risk': 'steelblue', 'Medium Risk': 'gold', 'High Risk': 'coral'}
tier_summary['users'].plot(kind='bar', ax=ax,
    color=[colors_tier[t] for t in tier_summary.index], edgecolor='white')
ax.set_title('Users by Risk Tier', fontweight='bold')
ax.set_ylabel('User Count')
ax.set_xticklabels(tier_summary.index, rotation=0)
plt.tight_layout()
plt.show()

## 10. Risk Segmentation

Assign users to risk tiers so the retention team can prioritise outreach.

In [ ]:
# Score all users
Xall = X.values
best_model_all = results[best_name]['model']

if best_res['use_scaled']:
    Xall = scaler.transform(Xall)

all_proba = best_model_all.predict_proba(Xall)[:, 1]

df_scored = df[['user_id', 'plan', 'tenure_months', 'churned']].copy()
df_scored['churn_probability'] = all_proba.round(4)

df_scored['risk_tier'] = pd.cut(
    df_scored['churn_probability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

tier_summary = df_scored.groupby('risk_tier', observed=True).agg(
    users       = ('user_id', 'count'),
    actual_churn= ('churned', 'mean'),
    avg_prob    = ('churn_probability', 'mean')
).round(3)

print(tier_summary)

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
colors_tier = {'Low Risk': 'steelblue', 'Medium Risk': 'gold', 'High Risk': 'coral'}
tier_summary['users'].plot(kind='bar', ax=ax,
    color=[colors_tier[t] for t in tier_summary.index],
    edgecolor='white')
ax.set_title('Users by Risk Tier', fontweight='bold')
ax.set_ylabel('User Count')
ax.set_xticklabels(tier_summary.index, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 highest-risk users to action
top_at_risk = (
    df_scored[df_scored['risk_tier'] == 'High Risk']
    .sort_values('churn_probability', ascending=False)
    .head(20)
)
print('Top 20 users by churn risk:')
top_at_risk

## 11. Business Insights & Recommended Actions

| Driver | Observation | Retention Action |
|---|---|---|
| **Skip rate** | High skippers churn significantly more | Improve recommendation engine; trigger a "Taste Refresh" prompt |
| **Active days** | Inactivity is a strong early warning | Push re-engagement notifications after 5 idle days |
| **Payment failures** | Each failure sharply increases churn risk | Auto-retry with grace period; prompt payment update in-app |
| **Tenure** | New users (≤3 months) are most vulnerable | Onboarding flow: guide playlist creation, suggest Discovery Weekly |
| **Plan = free** | Free-tier users churn at highest rate | Time-limited premium trial; targeted upgrade offer |
| **Social score** | Connected users are more sticky | Encourage friend follows; collaborative playlist features |

### Model Deployment Recommendation

- **Batch scoring:** Run the model weekly on all active users; feed scores into the CRM.
- **Threshold:** Use `optimal_t` (~0.35–0.40) to maximise recall — it's cheaper to over-notify than to lose a subscriber.
- **High-risk tier:** Trigger personalised push notifications, in-app messages, or a discount offer.
- **Monitor:** Track AUC and churn rate monthly; retrain quarterly or when distribution shifts.

In [ ]:
print('=== Final Model Summary ===')
print(f'Best model  : {best_name}')
print(f'ROC-AUC     : {best_res["roc_auc"]:.4f}')
print(f'PR-AUC      : {best_res["pr_auc"]:.4f}')
print(f'F1 (opt. t) : {f1_score(y_test, pred_opt):.4f}')
print(f'Threshold   : {optimal_t:.2f}')
print(f'High-risk users identified: {(df_scored["risk_tier"] == "High Risk").sum()}')